# Supplier Selection Scorecard: FinTech Data Science & Analytics
**Course:** FinTech Data Science and Analytics (7MD470)  
**Institution:** Walchand College of Engineering, Sangli  
**Guide:** Prof. Bipin Mahadik  
**Team:** Hrishikesh Bankapur, Alok Kulkarni, Trusha Kulkarni, Sumit Ashatapawar, Sahil Jagdale  

---
## Project Overview
This notebook reproduces the complete analysis, tables, and figures published in the project report:
1. **Data Cleaning & Preprocessing:** Handles 10,324 shipment records and non-numeric values.
2. **Engineered Metrics:** Cost Efficiency (Price Index, Freight Ratio), Delivery Reliability (On-Time Rate, Avg Delay), and Supply Chain Resilience (Delay Spread, Monthly Consistency).
3. **Multi-Criteria Scoring:** Min-Max scaling, Pillar scoring, and User-weighted Composite scoring.
4. **Sensitivity Analysis & Baseline Comparison:** Testing ranking stability across weight scenarios and proving why price-only selection fails.
5. **Advanced Modules:** Analytic Hierarchy Process (AHP), TOPSIS ranking, FinTech Working Capital & Safety Stock holding cost modeling, Sourcing Order Allocation (Linear Programming), and Temporal Backtesting.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from analysis import (
    load_and_clean_data,
    compute_supplier_metrics,
    normalize_and_score,
    run_sensitivity_analysis,
    ahp_weighting,
    topsis_ranking,
    compute_fintech_inventory_risk,
    optimize_order_allocation,
    run_backtesting_validation,
    generate_all_figures
)

print("Libraries loaded successfully.")

## 1. Load and Clean Dataset
We clean the USAID Supply Chain Shipment Pricing Data (10,324 records).

In [ ]:
raw_df, valid_df = load_and_clean_data("sample-data.csv")
df_metrics, excluded = compute_supplier_metrics(valid_df, min_shipments=20)
print(f"Ranked suppliers: {len(df_metrics)}, Excluded (<20 shipments): {len(excluded)}")

## 2. Table 8: Full Supplier Scorecard (Equal Weights 1:1:1)

In [ ]:
scored_df = normalize_and_score(df_metrics)
display_cols = ["rank", "name", "n", "composite", "cost", "reliability", "resilience", "onTime", "avgDelay"]
scorecard_table = scored_df[display_cols].copy()
scorecard_table["onTime"] = (scorecard_table["onTime"] * 100).round(1).astype(str) + "%"
scorecard_table = scorecard_table.round({"composite": 1, "cost": 1, "reliability": 1, "resilience": 1, "avgDelay": 1})
scorecard_table.head(10)

## 3. Table 9: Six Cheapest Vendors vs. Scorecard Rank
Demonstrates how suppliers with low sticker prices fall down when reliability is considered.

In [ ]:
scored_df["price_rank"] = scored_df["pidx"].rank(ascending=True).astype(int)
cheapest_table = scored_df.sort_values("price_rank").head(6)[["name", "pidx", "price_rank", "rank", "onTime", "avgDelay"]].copy()
cheapest_table["onTime"] = (cheapest_table["onTime"] * 100).round(1).astype(str) + "%"
cheapest_table = cheapest_table.round({"pidx": 2, "avgDelay": 1})
cheapest_table

## 4. Sensitivity Analysis Matrix (Table 6 / Figure 8)

In [ ]:
sens_df = run_sensitivity_analysis(df_metrics)
sens_df.head(10)

## 5. Advanced MCDM: AHP & TOPSIS

In [ ]:
ahp_w, ahp_cr = ahp_weighting()
print(f"AHP Calculated Weights: Cost={ahp_w['cost']:.2f}, Reliability={ahp_w['reliability']:.2f}, Resilience={ahp_w['resilience']:.2f} (CR: {ahp_cr:.3f})")
topsis_df = topsis_ranking(df_metrics)
topsis_df[["topsis_rank", "name", "topsis_score"]].head(5)

## 6. FinTech Impact: Working Capital & Safety Stock Carrying Cost
Quantifies the monetary cost of late deliveries and high variance.

In [ ]:
fin_df = compute_fintech_inventory_risk(scored_df)
fin_df[["rank", "name", "composite", "safety_stock_units", "working_capital_locked", "annual_holding_cost", "true_landed_cost"]].head(5)

## 7. Order Allocation Optimization (Linear Programming)

In [ ]:
alloc_df, ok = optimize_order_allocation(scored_df, total_order=100000, min_reliability=0.96)
alloc_df[["name", "allocated_units", "allocation_pct", "onTime", "pidx"]]

## 8. Generate and Display All Report Figures

In [ ]:
generate_all_figures(raw_df, scored_df, sens_df, fin_df)
print("All figures successfully saved to figures/ folder.")